# 03 — Heartbreak: The 2023 ODI World Cup

India went undefeated through the entire group stage, then lost the final to Australia on November 19, 2023. This notebook analyzes how Reddit reacted — before, during, and after the heartbreak.

### 1. Setup

In [ ]:
import sys
sys.path.insert(0, "..")

from utils import (
    get_spark, load_cricket_submissions, load_cricket_comments,
    add_player_mentions, add_time_features, label_event_period,
    save_figure, save_pandas_to_local,
    EVENT_DATES,
)

from pyspark.sql import functions as F
from pyspark.sql.functions import col, count, avg, sum as ssum, when, lower, to_date
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
import seaborn as sns
import pandas as pd

sns.set_theme(style="darkgrid")
plt.rcParams["figure.dpi"] = 120

### 2. Start Spark

In [ ]:
spark = get_spark("03_Heartbreak2023WC")
spark

### 3. Load filtered data

In [ ]:
subs = load_cricket_submissions(spark)
coms = load_cricket_comments(spark)
print("Loaded submissions and comments")

### 4. Add time features and event period labels

In [ ]:
subs = add_time_features(subs, ts_col="created_utc")
subs = label_event_period(subs, ts_col="created_dt")

coms = add_time_features(coms, ts_col="created_utc")
coms = label_event_period(coms, ts_col="created_dt")

### 5. Narrow to WC 2023 window

In [ ]:
wc_subs = subs.filter(
    (col("created_dt") >= EVENT_DATES["wc2023_start"]) &
    (col("created_dt") <= EVENT_DATES["wc2023_final_end"])
)
wc_coms = coms.filter(
    (col("created_dt") >= EVENT_DATES["wc2023_start"]) &
    (col("created_dt") <= EVENT_DATES["wc2023_final_end"])
)
print(f"WC 2023 submissions: {wc_subs.count():,}")
print(f"WC 2023 comments:    {wc_coms.count():,}")

### 6. Daily post volume

In [ ]:
daily_posts = (
    wc_subs
    .withColumn("date", to_date(col("created_dt")))
    .groupBy("date")
    .agg(count("*").alias("post_count"))
    .orderBy("date")
    .toPandas()
)
daily_posts["date"] = pd.to_datetime(daily_posts["date"])
print(daily_posts)

### 7. Line chart — daily post volume

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(daily_posts["date"], daily_posts["post_count"], color="#2a9d8f", linewidth=2, marker="o", markersize=4)
ax.axvline(EVENT_DATES["wc2023_final"], color="red", linestyle="--", linewidth=1.5, label="Final: India vs Australia (Loss)")
ax.fill_between(daily_posts["date"], daily_posts["post_count"], alpha=0.15, color="#2a9d8f")
ax.set_title("Daily Reddit Post Volume — 2023 ODI World Cup", fontsize=13)
ax.set_ylabel("Posts per Day")
ax.set_xlabel("Date")
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
plt.xticks(rotation=30)
plt.tight_layout()
save_figure(fig, "03_daily_posts_wc2023.png")
plt.show()

### 8. Player mentions during WC 2023

In [ ]:
wc_subs = add_player_mentions(wc_subs, text_col="title")
wc_coms = add_player_mentions(wc_coms, text_col="body")

wc_mentions = wc_subs.agg(
    ssum(col("mentions_dhoni").cast("int")).alias("Dhoni"),
    ssum(col("mentions_kohli").cast("int")).alias("Kohli"),
    ssum(col("mentions_rohit").cast("int")).alias("Rohit"),
).toPandas()
print("Player mentions during WC 2023:")
print(wc_mentions)

### 9. Player mentions by event period

In [ ]:
mentions_by_period = wc_subs.groupBy("event_period").agg(
    ssum(col("mentions_dhoni").cast("int")).alias("Dhoni"),
    ssum(col("mentions_kohli").cast("int")).alias("Kohli"),
    ssum(col("mentions_rohit").cast("int")).alias("Rohit"),
).toPandas()
print(mentions_by_period)
save_pandas_to_local(mentions_by_period, "03_wc2023_mentions_by_period.csv")

### 10. Grouped bar — mentions by period

In [ ]:
plot_df = mentions_by_period.set_index("event_period")[["Dhoni", "Kohli", "Rohit"]]
fig, ax = plt.subplots(figsize=(10, 5))
plot_df.plot(kind="bar", ax=ax, color=["#f4a261", "#2a9d8f", "#e76f51"], edgecolor="white", width=0.6)
ax.set_title("Player Mentions by Event Period — 2023 WC", fontsize=13)
ax.set_ylabel("Mentions")
ax.set_xlabel("")
ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha="right")
ax.legend(title="Player")
plt.tight_layout()
save_figure(fig, "03_wc2023_mentions_by_period.png")
plt.show()

### 11. Top posts after the final

In [ ]:
top_aftermath = (
    wc_subs
    .filter(col("event_period") == "wc2023_final_aftermath")
    .select("title", "subreddit", "score", "num_comments", "created_dt")
    .orderBy(col("score").desc())
    .limit(10)
    .toPandas()
)
print(top_aftermath[["title", "subreddit", "score", "num_comments"]].to_string(index=False))
save_pandas_to_local(top_aftermath, "03_wc2023_top_aftermath_posts.csv")

### 12. Engagement by period

In [ ]:
engagement_by_period = (
    wc_subs
    .groupBy("event_period")
    .agg(
        count("*").alias("posts"),
        avg("score").alias("avg_score"),
        avg("num_comments").alias("avg_comments"),
    )
    .toPandas()
)
print(engagement_by_period)
save_pandas_to_local(engagement_by_period, "03_wc2023_engagement_by_period.csv")

### 13. Bar chart — engagement by period

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors_period = ["#264653", "#e9c46a"]
n = len(engagement_by_period)

axes[0].bar(engagement_by_period["event_period"], engagement_by_period["avg_score"],
            color=colors_period[:n], edgecolor="white", width=0.5)
axes[0].set_title("Avg Post Score by Period")
axes[0].set_ylabel("Average Score")
axes[0].set_xticklabels(engagement_by_period["event_period"], rotation=20, ha="right")

axes[1].bar(engagement_by_period["event_period"], engagement_by_period["avg_comments"],
            color=colors_period[:n], edgecolor="white", width=0.5)
axes[1].set_title("Avg Comments per Post by Period")
axes[1].set_ylabel("Avg Comments")
axes[1].set_xticklabels(engagement_by_period["event_period"], rotation=20, ha="right")

plt.suptitle("Reddit Engagement — 2023 ODI World Cup", fontsize=13)
plt.tight_layout()
save_figure(fig, "03_wc2023_engagement_by_period.png")
plt.show()

### 14. Stop Spark

In [ ]:
spark.stop()